## Preprocess Data

In [3]:
# %%writefile ../build/movies_build.py
"""
movies_build.py
----------------
Preprocess the MovieLens 100k dataset and build FAISS index.
Run this script once to generate embeddings and save the index.
"""

import pandas as pd
from langchain_core.documents import Document
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_community.vectorstores import FAISS

def build_movies_index():
    # --- Step 1: Load raw MovieLens files ---
    movies = pd.read_csv(
        "../data/ml-100k/u.item",
        sep="|", encoding="latin-1",
        names=["movie_id","title","release_date","video_release_date","IMDb_URL",
               "unknown","Action","Adventure","Animation","Children","Comedy","Crime",
               "Documentary","Drama","Fantasy","Film-Noir","Horror","Musical","Mystery",
               "Romance","Sci-Fi","Thriller","War","Western"]
    )

    ratings = pd.read_csv(
        "../data/ml-100k/u.data",
        sep="\t", names=["user_id","item_id","rating","timestamp"]
    )

    users = pd.read_csv(
        "../data/ml-100k/u.user",
        sep="|", names=["user_id","age","gender","occupation","zip"]
    )

    # --- Step 2: Collapse one-hot genre flags into a genre_list ---
    genre_cols = ["Action","Adventure","Animation","Children","Comedy","Crime",
                  "Documentary","Drama","Fantasy","Film-Noir","Horror","Musical",
                  "Mystery","Romance","Sci-Fi","Thriller","War","Western"]

    def collapse_genres(row):
        return [g for g in genre_cols if row[g] == 1]

    movies["genre_list"] = movies.apply(collapse_genres, axis=1)

    # --- Step 3: Enrich with ratings (average + count) ---
    ratings_summary = ratings.groupby("item_id").agg(
        avg_rating=("rating","mean"),
        rating_count=("rating","count")
    ).reset_index().rename(columns={"item_id":"movie_id"})

    movies_enriched = movies.merge(ratings_summary, on="movie_id", how="left")

    # --- Step 4: Select lean schema ---
    movies_final = movies_enriched[[
        "movie_id","title","release_date","genre_list","avg_rating","rating_count"
    ]]

    # --- Step 5: Convert rows into LangChain Documents ---
    movie_docs = []
    for _, row in movies_final.iterrows():
        text = f"""
        Movie: {row['title']}
        Release Year: {row['release_date']}
        Genres: {', '.join(row['genre_list'])}
        Average Rating: {row['avg_rating']}
        Rating Count: {row['rating_count']}
        """
        movie_docs.append(Document(page_content=text.strip()))

    print(f"✅ Prepared {len(movie_docs)} movie documents.")

    # --- Step 6: Build embeddings + FAISS index ---
    embeddings = HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2")
    vectorstore = FAISS.from_documents(movie_docs, embeddings)

    # --- Step 7: Save index ---
    vectorstore.save_local("../data/faiss_movies_index")
    print("✅ Movies FAISS index built and saved at ../data/faiss_movies_index")

if __name__ == "__main__":
    build_movies_index()


✅ Prepared 1682 movie documents.
✅ Movies FAISS index built and saved at ../data/faiss_movies_index


## Validate Data

In [ ]:
# Total rows
total_rows = len(movies_final)

# Count missing and non-missing per column
missing_count = movies_final.isnull().sum()
non_missing_count = total_rows - missing_count
missing_percent = (missing_count / total_rows) * 100

# Combine into one DataFrame
missing_summary = pd.DataFrame({
    "non_missing": non_missing_count,
    "missing": missing_count,
    "missing_percent": missing_percent.round(2)
})

print(missing_summary)

## Load Variables

In [1]:
# load API key from .env 
from dotenv import load_dotenv, find_dotenv 
load_dotenv(find_dotenv())
# llm model
model="gpt-4o-mini"

## Business Logic

In [3]:
# %%writefile ../src/chain_movies.py
"""
chain_movies.py
----------------
Core LangChain module for MoviesRAG.

Responsibilities:
- Load the persisted FAISS index (built once in build/movies_build.py)
- Define a retrieval-augmented generation (RAG) chain for movie queries
- Designed for integration into the CultRAG pipeline
"""

# --- BUSINESS LOGIC ---

# Core LangChain modules
from langchain_openai import ChatOpenAI
from langchain_core.prompts import ChatPromptTemplate

# Embeddings + Vectorstore
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_community.vectorstores import FAISS
from utils.paths import DATA_DIR

# Step 1: Load persisted FAISS index (built once in build/movies_build.py)
embeddings = HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2")
vectorstore = FAISS.load_local(
    str(DATA_DIR / "faiss_movies_index"),
    embeddings,
    allow_dangerous_deserialization=True
)
retriever = vectorstore.as_retriever()

# Step 2: LLM → define the language model
llm = ChatOpenAI(model="gpt-4o-mini", temperature=0)

# Step 3: Prompt Template → instructions for how the assistant should answer
prompt = ChatPromptTemplate.from_template("""
You are a retrieval‑augmented assistant (RAG). 
Use ONLY the provided context to answer the question. 
If the answer is not in the context, say "Not found in the catalog."

Conversation so far:
{history}

Context:
{context}

Question:
{question}

Rules:
- Do NOT add items that are not in the context.
- Do NOT guess or hallucinate.
- Output a valid markdown table with headers.
- Include a short summary after the table.

Answer:
""")

# Step 4: Helper to format retrieved docs into a single string
def format_docs(docs):
    """
    Convert a list of LangChain Document objects into a single string.
    Each document’s page_content is joined with double newlines.
    Used to feed retrieved context into the prompt.
    """
    return "\n\n".join([d.page_content for d in docs])

# Step 5: Build the Core LCEL Retrieval Chain
chain_movies = (
    {
        "context": lambda x: format_docs(retriever.invoke(x["question"])),
        "question": lambda x: x["question"],
        "history": lambda x: x.get("history", "")
    }
    | prompt
    | llm
)


Overwriting ../src/chain_movies.py


## Query & Display

In [8]:
# Display helpers for Jupyter
from IPython.display import display, Markdown

# --- 8. Query ---
query = "Please list 3 good sci fi movies"

response = chain_movies.invoke({"question": query})

# --- 9. Display ---
display(Markdown(response.content))

| Movie                          | Release Year | Average Rating |
|--------------------------------|--------------|-----------------|
| Return of the Jedi            | 14-Mar-1997  | 4.0079          |
| Mystery Science Theater 3000: The Movie | 19-Apr-1996  | 3.4308          |
| Lost in Space                  | 27-Mar-1998  | 3.1667          |

These three sci-fi movies include "Return of the Jedi," which has the highest average rating among the listed films, followed by "Mystery Science Theater 3000: The Movie" and "Lost in Space."